# stardelt warehouse quickstart

This notebook is already wired to the platform: it queries the `warehouse` catalog through
Trino **as the person signed in to JupyterHub** — not as a service account, and with no
credential stored in this notebook or in the image.

* **Your identity.** JupyterHub signs you in through Keycloak and hands this server *your*
  access token when it starts (`TRINO_TOKEN`). Trino checks that token against the realm's
  keys and runs every statement as your username.
* **Your tables.** `warehouse` is an Apache Iceberg catalog served by Lakekeeper; the data
  is Parquet on the platform's S3. Every engine that speaks this catalog sees the same
  tables — and the same permissions, because the policy attaches to the token, not to the
  tool.

The sample table `covid.covid_summary` is tiny on purpose. Replace the SQL with your own.

In [ ]:
import base64
import json
import os
from urllib.parse import urlsplit

import pandas as pd
from trino.auth import JWTAuthentication
from trino.dbapi import connect

TRINO_URL = os.environ.get("TRINO_URL", "https://trino.alpha.dev.stardelt.io:443")
TOKEN = os.environ.get("TRINO_TOKEN")
HUB_USER = os.environ.get("JUPYTERHUB_USER")


def token_username(token):
    """The username inside the JWT, without verifying it (Trino verifies)."""
    if not token or token.count(".") != 2:
        return None
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)
    return json.loads(base64.urlsafe_b64decode(payload)).get("preferred_username")


# Trino denies a request whose declared user differs from the token's
# ("user X cannot impersonate user Y"), so the client user is the token's own.
TRINO_USER = HUB_USER or token_username(TOKEN) or "notebook"

# Over HTTPS Trino accepts the bearer token as-is, so the notebook needs no
# password and no secret: the hub copies the signed-in user's token in at spawn.
_url = urlsplit(TRINO_URL)
conn = connect(
    host=_url.hostname,
    port=_url.port or (443 if _url.scheme == "https" else 8080),
    http_scheme=_url.scheme,
    auth=JWTAuthentication(TOKEN) if TOKEN else None,
    user=TRINO_USER,
    source="stardelt-warehouse-quickstart",
)


def sql(query):
    """Run `query` and return its rows as a pandas DataFrame."""
    cur = conn.cursor()
    cur.execute(query)
    columns = [d[0] for d in (cur.description or [])]
    return pd.DataFrame(cur.fetchall(), columns=columns)


print("Trino:    ", TRINO_URL)
print("Hub user: ", HUB_USER or f"(not running under JupyterHub; token says {TRINO_USER})")
if not TOKEN:
    print("TRINO_TOKEN is not set — the queries below will be refused with 401.")

In [ ]:
me = sql("SELECT current_user AS signed_in_as")
current_user = me.loc[0, "signed_in_as"]
print(me.to_string(index=False))

if HUB_USER:
    assert current_user == HUB_USER, (
        f"Trino authenticated this session as {current_user!r}, "
        f"but the hub user is {HUB_USER!r}"
    )
    print(f"\nOK: Trino runs as {current_user!r} — the token in TRINO_TOKEN is yours.")

In [ ]:
schemas = sql("SHOW SCHEMAS FROM warehouse")
print(schemas.to_string(index=False))

In [ ]:
summary = sql(
    "SELECT country, confirmed_cases, total_deaths FROM warehouse.covid.covid_summary "
    "ORDER BY confirmed_cases DESC"
)
print(f"warehouse.covid.covid_summary — {len(summary)} rows")
print(summary.to_string(index=False))

In [ ]:
# The helper package this image ships (images/jupyter/stardelt/): the same
# warehouse through two calls, without the connection setup in the first cell.
import stardelt

me = stardelt.sql("SELECT current_user AS signed_in_as")
assert me.loc[0, "signed_in_as"] == current_user, me
print(f"stardelt.sql runs as {current_user!r} — the same identity as above.")

# Governed write: a 3-row frame round-trips through warehouse.covid on a
# table that is dropped again, so re-running this notebook starts clean.
smoke = pd.DataFrame({"id": [1, 2, 3], "note": ["a", "b", "c"]})
stardelt.write_df(smoke, "warehouse.covid.quickstart_smoke", mode="overwrite")
back = stardelt.sql("SELECT id, note FROM warehouse.covid.quickstart_smoke ORDER BY id")
pd.testing.assert_frame_equal(back, smoke)
stardelt.sql("DROP TABLE warehouse.covid.quickstart_smoke")
print(f"stardelt.write_df round-trip OK: {len(back)} rows written, read back, table dropped.")

## What just happened

`SELECT current_user` is your Keycloak username, not a shared account. That is the platform's
"one identity" rule in practice: the same token that opened this notebook is the token Trino
authorises, so row and column policy (when it is configured) applies to your queries, and the
audit trail names you.

`warehouse` is Iceberg through Lakekeeper. Ask for `SHOW TABLES FROM warehouse.<schema>` to see
the rest — anything created by an ingestion pipeline or another notebook lands there too.

In [ ]:
# Spark Connect — the catalogs you may see, with your token. Spark is heavy
# compute and optional in an install (`make install-spark`); when it is missing
# this cell says what it found instead of failing the notebook. Point
# SPARK_REMOTE elsewhere through the hub's singleuser.extraEnv if you moved it.
import socket
from urllib.parse import urlsplit

import stardelt

SPARK_REMOTE = os.environ.get(
    "SPARK_REMOTE", "sc://spark-connect.stardelt.svc.cluster.local:15002"
)
_spark = urlsplit(SPARK_REMOTE.replace("sc://", "http://"))
try:
    # A short TCP probe first: a client pointed at a blackholed endpoint
    # retries for a long time, and a quickstart should not hang.
    socket.create_connection((_spark.hostname, _spark.port or 15002), timeout=2).close()

    # The helper asks Trino which catalogs you may see and configures them
    # with your token; Lakekeeper decides what the session may read.
    spark = stardelt.spark()
    print("connected to Spark Connect at", SPARK_REMOTE)
    summary = spark.sql("SELECT count(*) AS countries FROM warehouse.covid.covid_summary")
    summary.show(truncate=False)
    print(f"Spark read warehouse.covid.covid_summary: {summary.first()['countries']} rows — the warehouse through a second engine")
except Exception as exc:  # noqa: BLE001 - Spark is an optional component
    print(f"Spark Connect is not reachable at {SPARK_REMOTE}: {type(exc).__name__}: {exc}")
    print("Known causes: Spark is not installed (`make install-spark`), the singleuser")
    print("NetworkPolicy lacks the egress rule (helm-values/jupyterhub.yaml), or the")
    print("token predates `make sync-catalog-audience` — restart the server and sign in again.")

## What this notebook can and cannot do today

* **Can:** `SHOW` and `SELECT` against every catalog Trino exposes, with your own identity;
  write to `warehouse` when the table and your groups allow it — and read the same Iceberg
  tables through Spark Connect, a second engine.
* **TLS, in-cluster.** With the platform IdP installed, `TRINO_URL` is the coordinator's
  in-cluster HTTPS endpoint (`https://trino.stardelt.svc.cluster.local:8443`) and
  `REQUESTS_CA_BUNDLE` is the internal CA that signs it — a real anchor, not a disabled
  certificate check. Nothing goes through the public ingress, and a plain-HTTP statement is
  refused, so the token is the only way in.
* **Token lifetime:** the hub's `jupyterhub` client issues an 8-hour token, placed in this
  server's environment at spawn. When it expires, restart the server from the hub — the hub
  refreshes your token before every spawn. After the Keycloak SSO session ends (10 hours),
  sign in again.
* **Spark:** the cell above opens a session against the ClusterIP-only Spark Connect endpoint
  and reads `warehouse.covid.covid_summary` through the catalogs Trino lists for you, with your
  token. Spark is optional: without `make install-spark` the cell says what it found.* **The helper:** `import stardelt` (shipped in the image, `images/jupyter/stardelt/`) gives
  `sql` and `write_df` — the queries above without the setup — plus `spark()` for bulk work.
  Catalog access is your identity; the S3 key stays the platform's, and row/column masks are
  Trino-only (design `multitenant-catalogs.md` §5.8).